# Combine candidate mappings across networks

This notebook combines every `network_<N>/net<N>_candidate_mappings.csv` file into one review table. It also adds `standard_name`, `cell_method`, `long_description`, and `display_name` from the database `meta_vars` table.

The source CSV files are only read; they are never changed. The combined CSV is replaced only after all validation succeeds. Run the notebook from top to bottom using the project `.venv` kernel.

## 1. Imports and project location

The project root is detected when Jupyter starts in the repository root or one folder below it. The shared loader is imported only after its `func/` folder is added to Python's search path.

In [9]:
import os
from pathlib import Path
import re
import sys
import tempfile

import pandas as pd
import sqlalchemy as sa


# Find the project before importing its shared loader.
search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (root for root in search_roots
     if (root / 'func' / 'func_variable_mapping.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        'Cannot find variable_mapping/func/func_variable_mapping.py. '
        'Start Jupyter from the variable_mapping project folder.'
    )
func_dir = str(PROJECT_ROOT / 'func')
if func_dir not in sys.path:
    sys.path.insert(0, func_dir)
from func_variable_mapping import load_candidate_mappings


## 2. Settings

Normally, these paths do not need to change. Set the `CRMP_DB_URL` environment variable if you need a different database connection. The default passfile location is derived from the repository path, so it works in both `/workspaces/crmprtd` and a local checkout.

In [10]:
NETWORKS_DIR = PROJECT_ROOT / 'networks'
OUTPUT_PATH = NETWORKS_DIR / 'all_networks_candidate_mappings.csv'

# Keep credentials outside the notebook. SQLAlchemy/psycopg2 reads the
# password from the repository-level .pgpass file.
passfile = PROJECT_ROOT.parents[1] / '.pgpass'
DB_URL = os.getenv(
    'CRMP_DB_URL',
    'postgresql+psycopg2://tongli1997@'
    'proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca/crmp'
    '?keepalives=1&keepalives_idle=300&keepalives_interval=300'
    f'&keepalives_count=9&passfile={passfile}',
)

print(f'Input folder: {NETWORKS_DIR}')
print(f'Output file:  {OUTPUT_PATH}')

Input folder: /workspaces/crmprtd/Quanlity_Control/variable_mapping/networks
Output file:  /workspaces/crmprtd/Quanlity_Control/variable_mapping/networks/all_networks_candidate_mappings.csv


## 3. Combination and validation logic

Before saving, this function verifies folder, filename, and row network IDs; rejects duplicate candidate keys; requires one matching `meta_vars` row per candidate; and checks that the metadata belongs to the same network.

In [11]:
OUTPUT_COLUMNS = [
    'network_id',
    'canonical_variable',
    'vars_id',
    'net_var_name',
    'standard_name',
    'cell_method',
    'long_description',
    'display_name',
    'network_station_count',
    'source_time_scale',
    'daily_aggregation',
]
METADATA_COLUMNS = [
    'vars_id', 'metadata_network_id', 'standard_name', 'cell_method',
    'long_description', 'display_name',
]
SOURCE_PATTERN = re.compile(
    r'network_(?P<folder_id>\d+)/net(?P<file_id>\d+)_candidate_mappings\.csv$'
)


def combine_candidate_mappings(
    networks_dir: Path,
    output_path: Path,
    engine: sa.Engine,
) -> pd.DataFrame:
    """Validate, combine, enrich, and atomically save candidate mappings."""
    networks_dir = Path(networks_dir)
    output_path = Path(output_path)

    # Only network-prefixed source registries are inputs. This excludes the
    # combined output itself when the notebook is run again.
    source_paths = sorted(
        networks_dir.glob('network_*/net*_candidate_mappings.csv')
    )
    if not source_paths:
        raise ValueError(
            f'No candidate mapping files found under {networks_dir}.'
        )

    frames = []
    for path in source_paths:
        relative = path.relative_to(networks_dir).as_posix()
        match = SOURCE_PATTERN.fullmatch(relative)
        if match is None:
            raise ValueError(f'Unexpected candidate mapping path: {path}')

        folder_id = int(match.group('folder_id'))
        file_id = int(match.group('file_id'))
        if folder_id != file_id:
            raise ValueError(
                f'Folder and filename network IDs disagree for {path}: '
                f'{folder_id} != {file_id}.'
            )

        # The shared loader fills optional legacy columns when necessary.
        frame = load_candidate_mappings(path)
        row_ids = set(frame['network_id'].dropna().astype(int))
        if row_ids and row_ids != {folder_id}:
            raise ValueError(
                f'network_id values {sorted(row_ids)} in {path} do not match '
                f'network_{folder_id}.'
            )
        frames.append(frame[[
            'network_id', 'vars_id', 'net_var_name',
            'network_station_count', 'source_time_scale',
            'canonical_variable', 'daily_aggregation',
        ]])

    combined = pd.concat(frames, ignore_index=True)
    duplicate_key = ['network_id', 'vars_id', 'canonical_variable']
    duplicates = combined.duplicated(duplicate_key, keep=False)
    if duplicates.any():
        duplicate_rows = combined.loc[duplicates, duplicate_key]
        raise ValueError(
            'Duplicate candidate mapping keys found: '
            f'{duplicate_rows.to_dict(orient="records")}'
        )

    if combined.empty:
        combined = pd.DataFrame(columns=OUTPUT_COLUMNS)
    else:
        vars_ids = sorted(
            int(vars_id) for vars_id in combined['vars_id'].unique()
        )
        metadata_query = sa.text(
            """
            SELECT vars_id, network_id AS metadata_network_id,
                   standard_name, cell_method, long_description, display_name
            FROM meta_vars
            WHERE vars_id IN :vars_ids
            """
        ).bindparams(sa.bindparam('vars_ids', expanding=True))
        metadata = pd.read_sql(
            metadata_query, engine, params={'vars_ids': vars_ids}
        )

        duplicate_metadata = metadata.duplicated('vars_id', keep=False)
        if duplicate_metadata.any():
            duplicate_ids = sorted(
                int(vars_id) for vars_id in
                metadata.loc[duplicate_metadata, 'vars_id'].unique()
            )
            raise ValueError(
                f'Duplicate meta_vars rows for vars_ids: {duplicate_ids}'
            )

        metadata_vars_ids = {
            int(vars_id) for vars_id in metadata['vars_id'].unique()
        }
        missing_metadata = sorted(set(vars_ids) - metadata_vars_ids)
        if missing_metadata:
            raise ValueError(
                f'Missing meta_vars rows for vars_ids: {missing_metadata}'
            )

        combined = combined.merge(
            metadata[METADATA_COLUMNS], on='vars_id', how='left',
            validate='many_to_one', sort=False,
        )
        network_mismatch = combined[
            combined['network_id'].astype(int).ne(
                combined['metadata_network_id'].astype(int)
            )
        ]
        if not network_mismatch.empty:
            mismatches = network_mismatch[[
                'vars_id', 'network_id', 'metadata_network_id'
            ]].to_dict(orient='records')
            raise ValueError(
                f'Candidate and meta_vars network IDs disagree: {mismatches}'
            )
        combined = combined.drop(
            columns='metadata_network_id'
        )[OUTPUT_COLUMNS]

    combined = combined.sort_values(
        ['network_id', 'canonical_variable', 'vars_id', 'net_var_name'],
        kind='stable',
    ).reset_index(drop=True)

    # Write beside the destination first, then replace it atomically. A
    # validation or write failure therefore leaves the prior CSV untouched.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode='w', suffix='.csv', prefix=f'.{output_path.stem}_',
            dir=output_path.parent, delete=False,
        ) as temporary:
            temporary_path = Path(temporary.name)
            combined.to_csv(temporary, index=False)
        temporary_path.replace(output_path)
    finally:
        if temporary_path is not None and temporary_path.exists():
            temporary_path.unlink()

    return combined

## 4. Run and review the latest combined result

Running this cell replaces the previous combined CSV. The displayed summary and preview correspond to the latest run only.

In [12]:
engine = sa.create_engine(DB_URL, pool_pre_ping=True)
try:
    combined_candidates = combine_candidate_mappings(
        NETWORKS_DIR, OUTPUT_PATH, engine
    )
finally:
    engine.dispose()

print(
    f'Saved {len(combined_candidates)} rows from '
    f'{combined_candidates.network_id.nunique()} networks to {OUTPUT_PATH}'
)
display(combined_candidates.head(20))

Saved 136 rows from 21 networks to /workspaces/crmprtd/Quanlity_Control/variable_mapping/networks/all_networks_candidate_mappings.csv


,network_id,canonical_variable,vars_id,net_var_name,standard_name,cell_method,long_description,display_name,network_station_count,source_time_scale,daily_aggregation
0,1,daily_max_temperature,428,MAX_TEMP,air_temperature,time: maximum,Maximum daily temperature,Temperature (Max.),2792,daily,max
1,1,daily_min_temperature,427,MIN_TEMP,air_temperature,time: minimum,Minimum daily temperature,Temperature (Min.),2792,daily,min
2,1,precipitation_amount,429,ONE_DAY_PRECIPITATION,lwe_thickness_of_precipitation_amount,time: sum,Daily precipitation,Precipitation Amount,3445,daily,sum
3,1,precipitation_amount,430,ONE_DAY_RAIN,thickness_of_rainfall_amount,time: sum,Daily rainfall,Rainfall Amount,3293,daily,sum
4,1,snow_depth,432,SNOW_ON_THE_GROUND,surface_snow_thickness,time: point,Amount of snow on the ground,Surface Snow Depth (Point),2189,daily,fixed_hour
5,1,snowfall_amount,431,ONE_DAY_SNOW,thickness_of_snowfall_amount,time: sum,Daily snow accumulation,Snowfall Amount,3283,daily,sum
6,2,air_temperature,434,CURRENT_AIR_TEMPERATURE1,air_temperature,time: point,NaN,Temperature (Point),176,hourly,mean
7,2,air_temperature,436,CURRENT_AIR_TEMPERATURE2,air_temperature,time: point,NaN,Temperature (Point),118,hourly,mean
8,2,air_temperature,721,air_temp,air_temperature,time: point,NaN,Temperature (Mean),194,hourly,mean
9,2,daily_max_temperature,433,MAXIMUM_AIR_TEMPERATURE,air_temperature,time: maximum,NaN,Temperature (Max.),130,hourly,max
